# G06 — LearnMate: Evidence-Grounded Course AI Tutor
## M.Tech. AI–ML (First Year) | Course: Deep Learning Networks
**Assignment:** CCA2 and LCA-3 AI Frontier Application & Research Challenge  
**Research Theme:** Evidence Agreement Verification and Calibrated Conformal Abstention  
**Falsifiable Hypothesis $\mathcal{H}_1$:** *Augmenting dense retrieval with cross-encoder propositional NLI verification and temperature-scaled conformal abstention achieves $\Delta \text{AUROC} \ge 0.15$ on unanswerable course queries and reduces ungrounded hallucinations to $<5\%$ without degrading in-syllabus answer accuracy.*

### 1. Environment Setup & Strict Reproducibility
We initialize deterministic seeds (`seed = 42`) and configure PyTorch and Sentence Transformers.

In [ ]:
import os
import sys
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.config import RANDOM_SEED, CONFORMAL_ALPHA
from src.retriever import HybridRetriever
from src.evidence_verifier import EvidenceAgreementVerifier
from src.abstention_engine import CalibratedAbstentionEngine
from src.tutor_generator import TutorGenerator
from src.metrics import evaluate_predictions

# Set seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print(f'Reproducible environment initialized with seed {RANDOM_SEED}.')

### 2. Loading Course Syllabus Corpus (CourseKB-DLN) & Benchmark Splits
We load the 8 indexed modules from the M.Tech Deep Learning Networks curriculum and the stratified validation/test splits.

In [ ]:
with open('../data/processed/chunks_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)['chunks']
print(f'Loaded {len(corpus)} knowledge passages across 8 lecture modules.')

with open('../data/processed/val_calibration_split.json', 'r', encoding='utf-8') as f:
    val_items = json.load(f)['items']

with open('../data/processed/test_evaluation_split.json', 'r', encoding='utf-8') as f:
    test_items = json.load(f)['items']

print(f'Validation calibration items: {len(val_items)} | Test evaluation items: {len(test_items)}')

### 3. Conformal Calibration on Validation Split
We compute the empirical Evidence Agreement Scores on the validation split and determine the conformal quantile threshold $\tau^*$ under risk $\alpha = 0.05$.

In [ ]:
retriever = HybridRetriever(corpus)
verifier = EvidenceAgreementVerifier()
engine = CalibratedAbstentionEngine(alpha=CONFORMAL_ALPHA)
generator = TutorGenerator()

val_scores, val_labels = [], []
for item in val_items:
    hits = retriever.retrieve(item['question'], top_k=1)
    v_res = verifier.verify_agreement(hits[0]['text'], item['question'])
    val_scores.append(v_res['agreement_score'])
    val_labels.append(item['is_answerable'])

engine.calibrate(val_scores, val_labels)
print(f'Calibrated Abstention Threshold tau* = {engine.calibrated_threshold:.4f}')

### 4. Baseline Evaluation & Comparative Benchmarks
We evaluate four models on the held-out test split:
1. Parametric Small LLM (Zero-retrieval)
2. Standard Dense RAG (Cosine cutoff 0.50)
3. Hybrid RAG (No NLI verification)
4. Proposed LearnMate (Hybrid + NLI Verifier + Calibrated Conformal Abstention)

In [ ]:
df_baselines = pd.read_csv('../results/baseline_vs_proposed.csv')
display_cols = ['system_name', 'overall_decision_accuracy', 'abstention_auroc', 'abstention_f1', 'hallucination_rate_unanswerable', 'expected_calibration_error', 'latency_ms']
df_baselines[display_cols]

### 5. Adversarial Stress Testing Across 4 Regimes
Evaluating robustness against missing syllabus topics, out-of-domain queries, and near-miss distractors.

In [ ]:
df_stress = pd.read_csv('../results/stress_test_results.csv')
df_stress

### 6. Component-Wise Ablation Study
Isolating the empirical contributions of the NLI Verifier, Conformal Calibration, and BM25 Hybrid Fusion.

In [ ]:
df_ablation = pd.read_csv('../results/ablation_results.csv')
df_ablation

### 7. Publication Figure Displays
Displaying the generated research figures.

In [ ]:
from IPython.display import Image, display
display(Image(filename='../paper/figures/fig1_system_architecture.png'))
display(Image(filename='../paper/figures/fig2_calibration_curves.png'))
display(Image(filename='../paper/figures/fig3_roc_pr_abstention.png'))
display(Image(filename='../paper/figures/fig4_stress_test_comparison.png'))
display(Image(filename='../paper/figures/fig5_ablation_study.png'))

### 8. Interactive Query Demonstration
Testing live queries against the LearnMate pipeline.

In [ ]:
sample_queries = [
    'How does Nesterov Accelerated Gradient prevent oscillations?',
    'How does Batch Normalization eliminate the need for backpropagation?',
    'What differential equation governs continuous-depth Neural ODEs?',
    'How does the Taylor Rule calculate central bank monetary interest rates?'
]

for q in sample_queries:
    print('='*70)
    print(f'QUERY: {q}')
    hits = retriever.retrieve(q, top_k=1)
    v_res = verifier.verify_agreement(hits[0]['text'], q)
    dec = engine.evaluate_decision(v_res['agreement_score'], hits[0], v_res)
    resp = generator.generate_grounded_response(q, hits[0], dec, v_res)
    print(f'ACTION: {dec["action"]} | Refusal Type: {dec["refusal_type"]}')
    print(f'EAS: {v_res["agreement_score"]:.3f} | Confidence: {dec["calibrated_confidence"]*100:.1f}%')
    print(f'RESPONSE:\n{resp["response_text"]}\n')